# Huấn luyện ALS trên MovieLens 32M
Dùng dataset sạch ml32m-full-v1.zip trong 02_Dataset_Processed. Source lấy trực tiếp từ GitHub theo SOURCE_REF. Notebook không đo hiệu năng HDFS/Alluxio.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, shutil, subprocess, zipfile, time
from collections import deque

BASE = Path('/content/drive/MyDrive/BigData/DoAn_GoiYPhim_Alluxio')
PROJECT = Path('/content/movie-recommendation-alluxio')
REPOSITORY = 'https://github.com/iuzenaurora/movie-recommendation-alluxio.git'
SOURCE_REF = 'origin/main'  # Use a commit SHA for a reproducible experiment.
for name in ['01_Dataset_Raw','02_Dataset_Processed','03_Notebooks','04_Models','05_Experiments','06_Documents']:
    (BASE / name).mkdir(parents=True, exist_ok=True)
if not PROJECT.exists():
    subprocess.run(['git','clone',REPOSITORY,str(PROJECT)],check=True)
if not (PROJECT / '.git').is_dir():
    raise ValueError('PROJECT already exists without Git; select a new project directory.')
if subprocess.check_output(['git','-C',str(PROJECT),'status','--porcelain'],text=True).strip():
    raise ValueError('Source has local changes. Preserve them before selecting another revision.')
subprocess.run(['git','-C',str(PROJECT),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(PROJECT),'checkout','--detach',SOURCE_REF],check=True)
os.chdir(PROJECT)
java_home = Path('/usr/lib/jvm/java-17-openjdk-amd64')
if not java_home.exists():
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','openjdk-17-jre-headless'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pyspark==3.5.6','numpy','matplotlib'],check=True)
os.environ['JAVA_HOME'] = str(java_home)
os.environ['PATH'] = str(java_home / 'bin') + ':' + os.environ['PATH']
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYTHONPATH'] = str(PROJECT / 'src')
sys.path.insert(0,str(PROJECT / 'src'))
import numpy as np
print('Source commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('Python:',sys.version,'NumPy:',np.__version__)
subprocess.run(['java','-version'],check=True)

def extract_checked(path, destination):
    with zipfile.ZipFile(path) as archive:
        if archive.testzip() is not None:
            raise ValueError('Corrupt ZIP: ' + str(path))
        for name in archive.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Unsafe ZIP path')
        archive.extractall(destination)


In [ ]:
VERSION = 'ml32m-full-v1'
DATA = Path('/content/processed') / VERSION
if not (DATA / 'manifest.json').exists():
    saved = BASE / '02_Dataset_Processed' / (VERSION + '.zip')
    assert saved.exists(), 'Thiếu ZIP dữ liệu sạch từ notebook 02'
    local = Path('/content/dataset-for-ranking.zip')
    shutil.copy2(saved, local)
    DATA.parent.mkdir(parents=True, exist_ok=True)
    extract_checked(local, DATA.parent)
subprocess.run([sys.executable,'scripts/verify_dataset.py','--data',str(DATA)],check=True)
metadata = json.loads((DATA / 'manifest.json').read_text())
print('Dataset version:', metadata['datasetVersion'])
print('Rows:', metadata['sampleRows'])
assert metadata['sampleRows'] == metadata['metadataJoinedRows'], 'Đang dùng dữ liệu đã lấy mẫu'


In [ ]:
RANKS = [20]
REGS = [0.1, 0.2]  # Đổi thành [0.1] nếu chỉ muốn chạy cấu hình A.
MAX_ITER = 10
MIN_TRAIN_RATINGS = 100
RANKING_USERS = 200
MODEL_NAME = 'als-colab-' + time.strftime('%Y%m%d-%H%M%S')
MODEL_OUT = Path('/content/models') / MODEL_NAME
if MODEL_OUT.exists():
    raise FileExistsError('Dùng MODEL_NAME mới; không ghi đè mô hình cũ')
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
mem = {}
for line in Path('/proc/meminfo').read_text().splitlines():
    key, value = line.split(':',1)
    mem[key] = int(value.strip().split()[0])
DRIVER_MEMORY = '4g' if mem['MemAvailable'] / 1024**2 >= 8 else '2g'
print('Spark driver:', DRIVER_MEMORY, 'Configs:', len(RANKS)*len(REGS))


In [ ]:
import pyspark
submit = Path(pyspark.__file__).parent / 'bin' / 'spark-submit'
cmd = [str(submit),'--master','local[2]','--driver-memory',DRIVER_MEMORY,
       '--conf','spark.sql.shuffle.partitions=32','src/train.py',
       '--data',str(DATA),'--manifest',str(DATA/'manifest.json'),'--out',str(MODEL_OUT),
       '--ranks',*[str(x) for x in RANKS],'--regs',*[str(x) for x in REGS],
       '--max-iter',str(MAX_ITER),'--min-train-ratings',str(MIN_TRAIN_RATINGS),
       '--selection-metric','ndcg','--ranking-users',str(RANKING_USERS)]
LOG = Path('/content') / (MODEL_NAME + '-train.log')
started = time.perf_counter()
tail = deque(maxlen=30)
with LOG.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(cmd,cwd=PROJECT,stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT,text=True)
    for line in process.stdout:
        log.write(line)
        tail.append(line)
        if line.startswith(('Fitting ', 'Validation:')):
            print(line.rstrip(), flush=True)
    returncode = process.wait()
shutil.copy2(LOG, BASE/'05_Experiments'/LOG.name)
if returncode != 0:
    print(''.join(tail))
    raise RuntimeError('Train failed; log đã lưu trên Drive, không cần chạy lại ETL')
print('Training minutes:',round((time.perf_counter()-started)/60,2))


In [ ]:
subprocess.run([sys.executable,'scripts/validate_package.py','--model',str(MODEL_OUT)],check=True)
info = json.loads((MODEL_OUT/'manifest.json').read_text())
print(json.dumps(info['metrics'],indent=2,ensure_ascii=False))
print('Candidate policy:',info['candidatePolicy'])
(BASE/'05_Experiments'/(MODEL_NAME+'-metrics.json')).write_text(
    json.dumps(info['metrics'],indent=2,ensure_ascii=False),encoding='utf-8')


In [ ]:
destination = BASE/'04_Models'/(MODEL_NAME+'.zip')
if destination.exists():
    raise FileExistsError('ZIP mô hình đã có trên Drive; dùng tên mới hoặc giữ bản đã lưu')
archive = Path(shutil.make_archive(str(Path('/content')/MODEL_NAME),'zip',
                                  root_dir=MODEL_OUT.parent,base_dir=MODEL_OUT.name))
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
shutil.copy2(archive,destination)
assert destination.stat().st_size == archive.stat().st_size
print('Đã lưu:',destination)


## Diễn giải
- Chọn bằng NDCG@10 trên validation; hòa NDCG thì chọn RMSE thấp hơn.
- Chỉ đếm số đánh giá trong train để tạo tập ứng viên. ALS và popularity dùng cùng tập.
- Validation loại phim train đã xem; test loại train + validation. Phim phù hợp bị lọc hoặc chưa có vector vẫn nằm trong mẫu số Recall.
- Ranking dùng tối đa 200 người dùng theo seed 42; báo cáo số thật sự được đánh giá và phần danh mục bị loại. Không mô tả là đánh giá ranking toàn bộ người dùng.
- RMSE/MAE/coverage dùng tập test, trước refit. Không bảo đảm thắng baseline.
- candidate_movie_ids.json đi cùng model và có checksum. Cập nhật serving/api.py trước khi kích hoạt model để dùng đúng bộ lọc trên website.
- Batch retrain giữ danh sách ứng viên của model cha; để mở rộng danh sách phim ứng viên cần tạo phiên bản chính sách mới qua quy trình đánh giá.
- Đây là huấn luyện Colab CPU đơn máy; benchmark Alluxio/HDFS/Lustre phải thực hiện riêng với tham số cố định.
